In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from langchain.chat_models import init_chat_model
model = init_chat_model("meta-llama/llama-4-scout-17b-16e-instruct", model_provider="groq")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'Llama 4 Scout 17B', 'release_date': '2025-04-05', 'last_updated': '2025-04-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000297B10C0910>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000297B10C1310>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [9]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool   # ← add this

@tool                                    # ← decorate with @tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny"

# agent = create_agent(
#     model=ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct"),  # ← active model, own quota
#     tools=[get_weather],
#     system_prompt="You are a helpful assistant."
# )
model_with_tools = model.bind_tools([get_weather])
model_with_tools


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'Llama 4 Scout 17B', 'release_date': '2025-04-05', 'last_updated': '2025-04-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000297B10C0910>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000297B10C1310>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_

Tool execution loop

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"} ]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
# Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.
